# Module 4: AgentCore Evaluations

## Overview

In Module 3, we deployed the Product Catalog Agent to AgentCore with **full observability enabled** — OpenTelemetry traces and GenAI events are already flowing to CloudWatch. In this module, we set up **AgentCore Evaluations** to automatically assess agent performance using both **online** and **on-demand** evaluation methods.

**AgentCore Evaluations** provides automated assessment tools that:
- Use LLM-as-a-Judge techniques to evaluate agent responses
- Support 14 built-in evaluators covering quality, safety, and accuracy
- Integrate seamlessly with CloudWatch GenAI Observability

## Evaluation Types

| Type | Description | Use Case |
|------|-------------|----------|
| **Online Evaluation** | Continuously monitors live production traffic | Production monitoring, quality trends |
| **On-Demand Evaluation** | Targeted assessment of specific traces/spans | Issue investigation, testing custom evaluators |

## Built-in Evaluators

| Level | Evaluators |
|-------|------------|
| **Session** | GoalSuccessRate |
| **Trace** | Coherence, Conciseness, ContextRelevance, Correctness, Faithfulness, Harmfulness, Helpfulness, InstructionFollowing, Refusal, ResponseRelevance, Stereotyping |
| **Tool** | ToolParameterAccuracy, ToolSelectionAccuracy |

## What We'll Do

1. **Setup** — Load deployment info from Module 3 and configure prerequisites
2. **Create Evaluation IAM Role** — Permissions for Bedrock model access, CloudWatch read/write
3. **Online Evaluation** — Create continuous monitoring configuration
4. **Generate Test Data** — Invoke the agent to create evaluation data
5. **On-Demand Evaluation** — Evaluate specific traces programmatically
6. **Evaluate a Specific Trace** — Target a single trace for detailed assessment
7. **View Results** — Analyze evaluation results in CloudWatch
8. **Trace Exploration** — Explore OTEL trace structure, parse span events and tool calls
9. **CloudWatch Custom Dashboard** — Build a single pane of glass dashboard
10. **On-Demand Evaluation for Specific Traces** — Create evaluation jobs and compare online vs on-demand
11. **Summary** — Review what was accomplished

## Prerequisites

- Module 3 completed (Product Catalog Agent deployed with observability)
- CloudWatch Transaction Search enabled
- AWS credentials with AgentCore and CloudWatch permissions

**Note:** This notebook retrieves the Product Catalog Agent directly from AgentCore API — you do not need to run Module 3 in the same session.

### Time: ~30 minutes

## Step 1: Setup and Configuration

In [ ]:
# Dependencies are installed via uv sync in Module 00.
# If you skipped Module 00, run: !cd .. && uv sync --frozen

In [ ]:
import boto3
import json
import time
import uuid
from datetime import datetime, timezone, timedelta
from botocore.exceptions import ClientError

# Load deployment info from Module 3
try:
    %store -r deployment_info
    %store -r REGION
    print(f"Loaded deployment info from Module 3")
    print(f"  Region: {REGION}")
    print(f"  Runtime ARN: {deployment_info.get('runtime_arn', 'N/A')}")
    print(f"  OTEL Service Name: {deployment_info.get('otel_service_name', 'N/A')}")
    print(f"  User Pool ID: {deployment_info.get('user_pool_id', 'N/A')}")
    print(f"  User Client ID: {deployment_info.get('user_client_id', 'N/A')}")
except:
    print("Could not load deployment info from Module 3")
    print("Using default region — will discover agent from API")
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    deployment_info = None

# Initialize AWS clients
sts_client = boto3.client('sts', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)
logs_client = boto3.client('logs', region_name=REGION)
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
cognito_client = boto3.client('cognito-idp', region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()['Account']

print(f"\nConfiguration:")
print(f"  Account ID: {ACCOUNT_ID}")
print(f"  Region: {REGION}")

In [ ]:
# Retrieve the Product Catalog Agent from Module 3
# This agent was deployed with observability in Module 3

AGENT_RUNTIME_NAME = 'ecommerce_workshop_product_catalog_agent'

def get_product_catalog_agent():
    """
    Find the Product Catalog Agent runtime by name.
    Handles pagination to search through all agents.
    """
    print(f"Searching for Product Catalog Agent: {AGENT_RUNTIME_NAME}\n")
    
    # First try deployment_info from Module 3
    if deployment_info and deployment_info.get('runtime_arn'):
        runtime_id = deployment_info.get('runtime_id')
        try:
            details = agentcore_control_client.get_agent_runtime(agentRuntimeId=runtime_id)
            if details.get('status') in ['READY', 'ACTIVE']:
                print(f"  Found agent from Module 3 deployment info")
                return {
                    'name': details.get('agentRuntimeName', AGENT_RUNTIME_NAME),
                    'id': runtime_id,
                    'arn': deployment_info['runtime_arn'],
                    'status': details.get('status')
                }
        except Exception:
            pass
    
    # Fall back to searching by name
    try:
        next_token = None
        page_count = 0
        
        while True:
            page_count += 1
            params = {}
            if next_token:
                params['nextToken'] = next_token
            
            response = agentcore_control_client.list_agent_runtimes(**params)
            runtimes = response.get('agentRuntimes', [])
            
            print(f"  Page {page_count}: checking {len(runtimes)} agents...")
            
            for runtime in runtimes:
                runtime_name = runtime.get('agentRuntimeName', '')
                if runtime_name == AGENT_RUNTIME_NAME:
                    print(f"  Found Product Catalog Agent!")
                    return {
                        'name': runtime_name,
                        'id': runtime['agentRuntimeId'],
                        'arn': runtime['agentRuntimeArn'],
                        'status': runtime.get('status', 'UNKNOWN')
                    }
            
            next_token = response.get('nextToken')
            if not next_token:
                break
        
        print(f"  Agent not found after {page_count} page(s)")
        return None
    except Exception as e:
        print(f"  Error searching for agent: {e}")
        return None


AGENT_INFO = get_product_catalog_agent()

print("\n" + "=" * 60)
print("PRODUCT CATALOG AGENT FOR EVALUATION")
print("=" * 60)

if AGENT_INFO:
    AGENT_ID = AGENT_INFO['id']
    AGENT_ARN = AGENT_INFO['arn']
    AGENT_NAME = AGENT_INFO['name']
    
    # Derive OTEL service name (same logic as Module 3)
    if deployment_info and deployment_info.get('otel_service_name'):
        OTEL_SERVICE_NAME = deployment_info['otel_service_name']
    else:
        OTEL_SERVICE_NAME = AGENT_NAME.replace('_', '-')
    
    print(f"\n  Name: {AGENT_NAME}")
    print(f"  ID: {AGENT_ID}")
    print(f"  ARN: {AGENT_ARN}")
    print(f"  Status: {AGENT_INFO['status']}")
    print(f"  OTEL Service Name: {OTEL_SERVICE_NAME}")
else:
    print("\nERROR: Product Catalog Agent not found!")
    print("   Please ensure Module 3 was completed successfully.")
    print(f"   Looking for agent named: {AGENT_RUNTIME_NAME}")
    AGENT_ID = None
    AGENT_ARN = None
    OTEL_SERVICE_NAME = None

### Pre-flight Check: Verify Observability Data

Before proceeding with evaluations, let's verify that Module 3's observability setup is working — 
i.e., that CloudWatch log groups exist and contain data from agent invocations.


In [ ]:
# Pre-flight check: verify observability data exists
print("=" * 60)
print("PRE-FLIGHT CHECK: OBSERVABILITY DATA")
print("=" * 60)

preflight_ok = True

if AGENT_ID:
    check_groups = [
        f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT",
        "aws/spans",
    ]
    
    for lg_name in check_groups:
        try:
            resp = logs_client.describe_log_groups(logGroupNamePrefix=lg_name)
            groups = [g for g in resp.get("logGroups", []) if g["logGroupName"] == lg_name]
            if groups:
                stored = groups[0].get("storedBytes", 0)
                status = "✅ exists" + (f" ({stored:,} bytes)" if stored > 0 else " (empty)")
                if stored == 0:
                    preflight_ok = False
            else:
                status = "❌ NOT FOUND"
                preflight_ok = False
        except Exception as e:
            status = f"❌ Error: {str(e)[:60]}"
            preflight_ok = False
        print(f"  {lg_name}: {status}")

    if preflight_ok:
        print("\n✅ Observability data is available. Ready for evaluations.")
    else:
        print("\n⚠️  Some log groups are missing or empty.")
        print("This may mean:")
        print("  - Module 3 was not completed (deploy the agent first)")
        print("  - Log delivery was not configured (run Module 3 Step 10b)")
        print("  - Agent has not been invoked yet (run Module 3 Step 11)")
        print("  - Logs are still propagating (wait 2-3 minutes and re-run this cell)")
else:
    print("❌ Agent not found. Complete Module 3 first.")
    preflight_ok = False


In [ ]:
# Authenticate with Cognito to get JWT tokens for agent invocation
# The Product Catalog Agent requires bearer_token and access_token in the payload
# to authenticate with the Gateway and determine user role for RBAC

# Load Cognito configuration from Module 3 deployment info
USER_POOL_ID = deployment_info.get('user_pool_id') if deployment_info else None
USER_CLIENT_ID = deployment_info.get('user_client_id') if deployment_info else None
CUSTOMER_EMAIL = 'john.customer@example.com'  # Test user created in Module 3
TEST_PASSWORD = 'Workshop1234!'

# Fallback: discover user_client_id from Cognito if not in deployment_info
if USER_POOL_ID and not USER_CLIENT_ID:
    print("user_client_id not found in deployment_info, discovering from Cognito...")
    try:
        clients_resp = cognito_client.list_user_pool_clients(UserPoolId=USER_POOL_ID, MaxResults=60)
        for client in clients_resp.get('UserPoolClients', []):
            if 'user-client' in client.get('ClientName', ''):
                USER_CLIENT_ID = client['ClientId']
                print(f"  Found user client: {USER_CLIENT_ID} ({client['ClientName']})")
                break
        if not USER_CLIENT_ID:
            # Use the first client without a secret (user-facing clients don't have secrets)
            for client in clients_resp.get('UserPoolClients', []):
                details = cognito_client.describe_user_pool_client(
                    UserPoolId=USER_POOL_ID, ClientId=client['ClientId']
                )
                if not details['UserPoolClient'].get('ClientSecret'):
                    USER_CLIENT_ID = client['ClientId']
                    print(f"  Found user client (no secret): {USER_CLIENT_ID}")
                    break
    except Exception as e:
        print(f"  Error discovering client: {e}")

def get_user_tokens(user_pool_id, client_id, email, password):
    """Get JWT tokens for a Cognito user via ADMIN_USER_PASSWORD_AUTH."""
    try:
        response = cognito_client.admin_initiate_auth(
            UserPoolId=user_pool_id,
            ClientId=client_id,
            AuthFlow='ADMIN_USER_PASSWORD_AUTH',
            AuthParameters={
                'USERNAME': email,
                'PASSWORD': password
            }
        )
        tokens = response.get('AuthenticationResult', {})
        print(f"  Authenticated: {email}")
        return {
            'id_token': tokens.get('IdToken', ''),
            'access_token': tokens.get('AccessToken', ''),
        }
    except Exception as e:
        print(f"  Authentication error: {e}")
        return {}


print("Authenticating test user for agent invocation...\n")

if USER_POOL_ID and USER_CLIENT_ID:
    USER_TOKENS = get_user_tokens(USER_POOL_ID, USER_CLIENT_ID, CUSTOMER_EMAIL, TEST_PASSWORD)
    if USER_TOKENS.get('id_token'):
        print(f"  ID token obtained (length: {len(USER_TOKENS['id_token'])})")
        print(f"  Access token obtained (length: {len(USER_TOKENS['access_token'])})")
    else:
        print("  WARNING: Could not get tokens. Agent invocations may fail.")
        USER_TOKENS = {}
else:
    print("ERROR: Cognito configuration not found in deployment_info.")
    print("  Ensure Module 3 stored user_pool_id and user_client_id.")
    print("  You may need to re-run the Module 3 deployment_info store cell (cell-34).")
    USER_TOKENS = {}

## Step 2: Create IAM Role for Evaluations

AgentCore Evaluations requires an IAM role with permissions to:
- Invoke Amazon Bedrock models (for LLM-as-Judge evaluation)
- Read traces from Amazon CloudWatch
- Write evaluation results to Amazon CloudWatch

In [ ]:
EVALUATION_ROLE_NAME = 'ecommerce-workshop-evaluation-role'

def create_evaluation_role():
    """Create IAM role for AgentCore Evaluations."""
    print(f"Creating evaluation role: {EVALUATION_ROLE_NAME}\n")
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID}
            }
        }]
    }
    
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "BedrockModelAccess",
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream"
                ],
                "Resource": [
                    f"arn:aws:bedrock:{REGION}::foundation-model/*",
                    f"arn:aws:bedrock:*:{ACCOUNT_ID}:inference-profile/*"
                ]
            },
            {
                "Sid": "CloudWatchLogsRead",
                "Effect": "Allow",
                "Action": [
                    "logs:GetLogEvents",
                    "logs:FilterLogEvents",
                    "logs:StartQuery",
                    "logs:GetQueryResults",
                    "logs:DescribeLogGroups",
                    "logs:DescribeLogStreams"
                ],
                "Resource": [
                    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:*"
                ]
            },
            {
                "Sid": "CloudWatchLogsWrite",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents"
                ],
                "Resource": [
                    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/evaluations/*"
                ]
            },
            {
                "Sid": "CloudWatchMetrics",
                "Effect": "Allow",
                "Action": ["cloudwatch:PutMetricData"],
                "Resource": "*",
                "Condition": {
                    "StringEquals": {
                        "cloudwatch:namespace": "Bedrock-AgentCore/Evaluations"
                    }
                }
            },
            {
                "Sid": "XRayRead",
                "Effect": "Allow",
                "Action": [
                    "xray:GetTraceSummaries",
                    "xray:BatchGetTraces",
                    "xray:GetTraceGraph"
                ],
                "Resource": "*"
            }
        ]
    }
    
    try:
        # Check if role already exists
        try:
            response = iam_client.get_role(RoleName=EVALUATION_ROLE_NAME)
            role_arn = response['Role']['Arn']
            print(f"Role already exists: {role_arn}")
            # Update permissions policy
            iam_client.put_role_policy(
                RoleName=EVALUATION_ROLE_NAME,
                PolicyName='evaluation-permissions',
                PolicyDocument=json.dumps(permissions_policy)
            )
            print("Updated permissions policy")
            return role_arn
        except ClientError as e:
            if e.response['Error']['Code'] != 'NoSuchEntity':
                raise
        
        # Create the role
        response = iam_client.create_role(
            RoleName=EVALUATION_ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='IAM role for AgentCore Evaluations'
        )
        role_arn = response['Role']['Arn']
        print(f"Created role: {role_arn}")
        
        iam_client.put_role_policy(
            RoleName=EVALUATION_ROLE_NAME,
            PolicyName='evaluation-permissions',
            PolicyDocument=json.dumps(permissions_policy)
        )
        print("Attached permissions policy")
        
        print("Waiting for role to propagate...")
        time.sleep(10)
        return role_arn
        
    except Exception as e:
        print(f"Error creating role: {e}")
        return None


EVALUATION_ROLE_ARN = create_evaluation_role()
print(f"\nEvaluation Role ARN: {EVALUATION_ROLE_ARN}")

## Step 3: Create Online Evaluation Configuration

Online evaluation continuously monitors agent quality using live production traffic. We create an evaluation configuration for the Product Catalog Agent.

**Configuration includes:**
- **Data source**: CloudWatch log groups from our observability-enabled agent
- **Evaluators**: Helpfulness, GoalSuccessRate, ToolSelectionAccuracy, Coherence
- **Sampling**: 100% (for demo purposes; use 10-20% in production)

In [ ]:
ONLINE_EVAL_CONFIG_NAME = 'ecommerce_workshop_product_catalog_eval'
ONLINE_EVAL_CONFIG = None

def get_agent_log_groups(agent_id, endpoint='DEFAULT'):
    """Get CloudWatch log groups for an agent."""
    return [
        f"/aws/bedrock-agentcore/runtimes/{agent_id}-{endpoint}",
        f"/aws/vendedlogs/bedrock-agentcore/{agent_id}"
    ]


if AGENT_ID and OTEL_SERVICE_NAME:
    LOG_GROUPS = get_agent_log_groups(AGENT_ID, 'DEFAULT')
    
    print("Agent Configuration for Evaluation:")
    print(f"  Agent ID: {AGENT_ID}")
    print(f"  OTEL Service Name: {OTEL_SERVICE_NAME}")
    print(f"  Log Groups:")
    for lg in LOG_GROUPS:
        print(f"    - {lg}")
else:
    print("ERROR: Agent not found. Please run Step 1 first.")
    LOG_GROUPS = []


# --- Create online evaluation configuration ---

def create_online_evaluation_config():
    """Create an online evaluation configuration for continuous monitoring."""
    print(f"\nCreating online evaluation config: {ONLINE_EVAL_CONFIG_NAME}\n")
    
    evaluators = [
        {"evaluatorId": "Builtin.Helpfulness"},
        {"evaluatorId": "Builtin.GoalSuccessRate"},
        {"evaluatorId": "Builtin.ToolSelectionAccuracy"},
        {"evaluatorId": "Builtin.Coherence"}
    ]
    
    # Service name format for AgentCore Runtime: <agent-runtime-name>.<endpoint-name>
    service_name = f"{AGENT_NAME}.DEFAULT"
    
    try:
        response = agentcore_control_client.create_online_evaluation_config(
            onlineEvaluationConfigName=ONLINE_EVAL_CONFIG_NAME,
            description="Online evaluation for Product Catalog Agent",
            rule={
                "samplingConfig": {"samplingPercentage": 100.0}  # 100% for demo; use 10-20% in production
            },
            dataSourceConfig={
                "cloudWatchLogs": {
                    "logGroupNames": LOG_GROUPS,
                    "serviceNames": [service_name]
                }
            },
            evaluators=evaluators,
            evaluationExecutionRoleArn=EVALUATION_ROLE_ARN,
            enableOnCreate=True
        )
        
        config_id = response.get('onlineEvaluationConfigId', 'unknown')
        print(f"Created online evaluation config!")
        print(f"  Config ID: {config_id}")
        print(f"  Status: {response.get('status', 'CREATING')}")
        print(f"  Execution Status: {response.get('executionStatus', 'N/A')}")
        
        return {
            'config_name': ONLINE_EVAL_CONFIG_NAME,
            'config_id': config_id,
            'status': response.get('status', 'CREATING'),
            'evaluators': [e['evaluatorId'] for e in evaluators]
        }
    except ClientError as e:
        error_code = e.response['Error']['Code']
        error_msg = e.response['Error']['Message']
        
        if 'ConflictException' in error_code or 'already exists' in error_msg.lower():
            print(f"Online evaluation config already exists, retrieving...")
            try:
                configs = agentcore_control_client.list_online_evaluation_configs()
                for config in configs.get('items', configs.get('onlineEvaluationConfigs', [])):
                    if config.get('onlineEvaluationConfigName') == ONLINE_EVAL_CONFIG_NAME:
                        config_id = config.get('onlineEvaluationConfigId')
                        print(f"  Found existing config: {config_id}")
                        return {
                            'config_name': ONLINE_EVAL_CONFIG_NAME,
                            'config_id': config_id,
                            'status': config.get('status', 'ACTIVE'),
                            'evaluators': [e['evaluatorId'] for e in evaluators]
                        }
            except Exception as list_err:
                print(f"  Error listing configs: {list_err}")
        else:
            print(f"Error creating config: {error_code} - {error_msg}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None


if AGENT_ARN and EVALUATION_ROLE_ARN and LOG_GROUPS:
    ONLINE_EVAL_CONFIG = create_online_evaluation_config()
    
    if ONLINE_EVAL_CONFIG:
        print(f"\nOnline Evaluation Config:")
        print(f"  Name: {ONLINE_EVAL_CONFIG['config_name']}")
        print(f"  ID: {ONLINE_EVAL_CONFIG['config_id']}")
        print(f"  Status: {ONLINE_EVAL_CONFIG['status']}")
        print(f"  Evaluators: {', '.join(ONLINE_EVAL_CONFIG['evaluators'])}")
    else:
        print("\nWARNING: Could not create online evaluation config.")
        print("On-demand evaluation will still work.")
else:
    print("ERROR: Missing prerequisites. Ensure Steps 1-2 completed successfully.")

#### Why This Module Uses Online Evaluation Only

> **Note:** This workshop does **not** use AgentCore's on-demand evaluation API.
>
> The on-demand Evaluate API expects spans from specific instrumentation scopes 
> (`strands.telemetry.tracer`, `opentelemetry.instrumentation.langchain`), but the 
> AgentCore Runtime's auto-instrumented OTEL traces use different scope names. This 
> scope mismatch means on-demand evaluation returns "no spans with supported scope" errors.
>
> **Online evaluation works correctly** because it hooks into the runtime pipeline directly, 
> bypassing the scope matching issue. For batch evaluation of specific traces, see **Module 05**, 
> which extracts OTEL events directly from CloudWatch logs instead of using the Evaluate API.

## Step 4: Generate Test Data

Invoke the Product Catalog Agent with test queries to generate traces for evaluation. The agent requires JWT tokens for Gateway authentication and RBAC.

In [ ]:
def invoke_agent(agent_arn, prompt, session_id=None):
    """
    Invoke the Product Catalog Agent and return the response.
    Includes JWT tokens required for Gateway authentication and RBAC.
    """
    if not session_id:
        # Generate session ID with min 33 chars (API requirement)
        session_id = f"eval-test-{int(time.time())}-{uuid.uuid4().hex}"
    
    print(f"\nInvoking agent...")
    print(f"  Prompt: {prompt[:80]}..." if len(prompt) > 80 else f"  Prompt: {prompt}")
    print(f"  Session ID: {session_id}")
    
    start_time = time.time()
    
    try:
        # Build payload with JWT tokens (same format as Module 3)
        payload = json.dumps({
            "prompt": prompt,
            "bearer_token": USER_TOKENS.get('id_token', ''),
            "access_token": USER_TOKENS.get('access_token', ''),
            "session_id": session_id
        })
        
        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=agent_arn,
            runtimeSessionId=session_id,
            payload=payload.encode('utf-8'),
            qualifier='DEFAULT'
        )
        
        elapsed = time.time() - start_time
        
        response_body = response.get('response', b'')
        if hasattr(response_body, 'read'):
            response_body = response_body.read()
        if isinstance(response_body, bytes):
            response_body = response_body.decode('utf-8')
        
        try:
            response_data = json.loads(response_body)
        except (json.JSONDecodeError, TypeError):
            response_data = response_body
        
        # Extract response text
        if isinstance(response_data, dict):
            message = response_data.get('response', response_data.get('message', str(response_data)))
        else:
            message = str(response_data)
        
        # Check for agent-level errors
        is_error = (isinstance(response_data, dict) and 
                    response_data.get('status') == 'error')
        
        print(f"  Response received in {elapsed:.2f}s")
        if is_error:
            print(f"  ERROR: {response_data.get('error', 'Unknown error')}")
        else:
            print(f"  Response: {str(message)[:120]}..." if len(str(message)) > 120 else f"  Response: {message}")
        
        return {
            'success': not is_error,
            'session_id': session_id,
            'elapsed_time': elapsed,
            'response': response_data,
            'message': message,
            'timestamp': datetime.now(timezone.utc).isoformat()
        }
    except Exception as e:
        print(f"  Error: {e}")
        return {
            'success': False,
            'session_id': session_id,
            'error': str(e),
            'timestamp': datetime.now(timezone.utc).isoformat()
        }

In [ ]:
# Test queries covering different product catalog scenarios
TEST_QUERIES = [
    "Find wireless headphones under $100",
    "What are the details for product PROD-001?",
    "Check inventory availability for PROD-003",
    "Compare products PROD-001 and PROD-002 side by side",
    "Recommend products in the Electronics category under $200",
    "Search for gaming accessories",
]

print("=" * 60)
print("GENERATING TEST DATA FOR EVALUATION")
print("=" * 60)
print(f"\nWill invoke {len(TEST_QUERIES)} queries to generate evaluation data")

INVOCATION_RESULTS = []
SESSION_IDS = []

if AGENT_ARN:
    print(f"\nAgent ARN: {AGENT_ARN}")
    print(f"Agent Status: {AGENT_INFO.get('status', 'UNKNOWN')}")
    
    for i, query in enumerate(TEST_QUERIES, 1):
        print(f"\n{'=' * 60}")
        print(f"Query {i}/{len(TEST_QUERIES)}")
        print(f"{'=' * 60}")
        
        result = invoke_agent(AGENT_ARN, query)
        INVOCATION_RESULTS.append(result)
        time.sleep(2)
    
    # Summary
    print("\n" + "=" * 60)
    print("INVOCATION SUMMARY")
    print("=" * 60)
    successful = sum(1 for r in INVOCATION_RESULTS if r.get('success'))
    print(f"\n{successful}/{len(INVOCATION_RESULTS)} invocations successful")
    
    SESSION_IDS = [r['session_id'] for r in INVOCATION_RESULTS if r.get('success')]
    print(f"Session IDs collected: {len(SESSION_IDS)}")
else:
    print("\nERROR: Agent not available. Please run Step 1 first.")

## Step 5: View Evaluation Results in CloudWatch

Evaluation results are stored in CloudWatch and can be viewed in:
1. **GenAI Observability Dashboard** — Aggregated scores and trends
2. **CloudWatch Metrics** — Evaluation score metrics
3. **CloudWatch Logs** — Detailed evaluation results

In [ ]:
def check_evaluation_log_group():
    """Check if evaluation results log group exists."""
    if not ONLINE_EVAL_CONFIG:
        return None
    config_id = ONLINE_EVAL_CONFIG.get('config_id')
    log_group = f"/aws/bedrock-agentcore/evaluations/results/{config_id}"
    try:
        logs_client.describe_log_groups(logGroupNamePrefix=log_group)
        return log_group
    except:
        return None


# Check evaluation results
print("=" * 60)
print("CLOUDWATCH EVALUATION RESULTS")
print("=" * 60)

eval_log_group = check_evaluation_log_group()

if eval_log_group:
    print(f"\nEvaluation log group found: {eval_log_group}")
    
    # Query for results
    try:
        query = "fields @timestamp, @message | filter @message like /evaluation/ | sort @timestamp desc | limit 20"
        start_time = datetime.now(timezone.utc) - timedelta(hours=1)
        end_time = datetime.now(timezone.utc)
        
        query_response = logs_client.start_query(
            logGroupName=eval_log_group,
            startTime=int(start_time.timestamp()),
            endTime=int(end_time.timestamp()),
            queryString=query
        )
        
        while True:
            result = logs_client.get_query_results(queryId=query_response['queryId'])
            if result['status'] in ['Complete', 'Failed']:
                break
            time.sleep(1)
        
        eval_logs = result.get('results', [])
        if eval_logs:
            print(f"Found {len(eval_logs)} evaluation result entries")
        else:
            print("No evaluation results yet. Online evaluation results will appear")
            print("after the evaluation job processes agent traces.")
    except Exception as e:
        print(f"Error querying logs: {e}")
else:
    print("\nEvaluation log group not found yet.")
    print("Online evaluation results will be created after the first evaluation runs.")

In [ ]:
# CloudWatch console links
print("\n" + "=" * 70)
print("CLOUDWATCH CONSOLE LINKS")
print("=" * 70)

print(f"\nGenAI Observability Dashboard (with Evaluations):")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")

print(f"\nCloudWatch Metrics (Evaluation Scores):")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#metricsV2:graph=~();namespace=Bedrock-AgentCore/Evaluations")

if ONLINE_EVAL_CONFIG:
    config_id = ONLINE_EVAL_CONFIG.get('config_id', '')
    print(f"\nEvaluation Results Log Group:")
    print(f"  /aws/bedrock-agentcore/evaluations/results/{config_id}")

print(f"\nX-Ray Traces:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:traces")

## Step 6: Trace Exploration — The OTEL "Aha Moment"

AgentCore uses **OpenTelemetry (OTEL)** to capture detailed traces of every agent invocation. This is where observability becomes tangible — you can see **exactly** how the agent thinks:

```
Trace (one user request)
├── Root Span (agent invocation)
│   ├── LLM Call Span (model inference)
│   │   └── Events: gen_ai.user.message → gen_ai.assistant.message
│   ├── Tool Selection Span
│   │   └── Event: gen_ai.assistant (with tool_use content)
│   ├── Tool Execution Span (e.g., search_products)
│   │   └── Event: gen_ai.tool.message (tool result)
│   └── LLM Call Span (final response generation)
│       └── Event: gen_ai.assistant.message (final answer)
```

**Why this matters:** Without OTEL traces, an agent is a black box — you see input and output but nothing in between. With traces, you see the full decision flow:

1. **What did the user ask?** → `gen_ai.user.message` event
2. **What did the LLM decide to do?** → `gen_ai.assistant` event (may contain `tool_use`)
3. **Which tool was called and with what parameters?** → Tool execution span + attributes
4. **What did the tool return?** → `gen_ai.tool.message` event
5. **How did the LLM synthesize the final answer?** → Final `gen_ai.assistant.message`

Each span includes:
- **Attributes**: metadata like `gen_ai.system`, `gen_ai.request.model`, `session.id`
- **Events**: detailed data such as `gen_ai.user.message`, `gen_ai.assistant.message`, `gen_ai.tool.message`
- **Timing**: start/end timestamps for latency analysis

**Key scopes** (instrumentation libraries that produce spans):
- `strands.telemetry.tracer` — Strands agent framework spans (final response, session metadata)
- `opentelemetry.instrumentation.botocore.bedrock-runtime` — LLM invocation spans (model calls, tool calls)

We can query these traces from CloudWatch Logs Insights to understand agent behavior at a granular level.

In [ ]:
# Query recent GenAI traces from CloudWatch Logs Insights
print("=" * 60)
print("FETCHING RECENT TRACES")
print("=" * 60)

query = '''
fields @timestamp, @message
| filter @message like /gen_ai/
| sort @timestamp desc
| limit 20
'''

# Search across known log groups for GenAI trace data
TRACE_LOG_GROUPS = [
    "aws/spans",
    f"/aws/vendedlogs/bedrock-agentcore/{AGENT_ID}" if AGENT_ID else None
]
TRACE_LOG_GROUPS = [lg for lg in TRACE_LOG_GROUPS if lg]

recent_traces = []

for log_group in TRACE_LOG_GROUPS:
    print(f"\nQuerying log group: {log_group}")
    try:
        results = query_logs(log_group, query, time_range_minutes=120)
        print(f"  Found {len(results)} entries")
        
        for row in results:
            for field in row:
                if field['field'] == '@message':
                    value = field['value'].strip()
                    if value.startswith('{'):
                        try:
                            span = json.loads(value)
                            recent_traces.append(span)
                        except json.JSONDecodeError:
                            pass
    except Exception as e:
        print(f"  Error: {str(e)[:80]}")

print(f"\nTotal GenAI trace spans collected: {len(recent_traces)}")
if recent_traces:
    # Show unique trace IDs
    trace_ids = set(t.get('traceId', 'unknown') for t in recent_traces)
    print(f"Unique trace IDs: {len(trace_ids)}")
    for tid in list(trace_ids)[:5]:
        print(f"  - {tid[:40]}...")

In [ ]:
# Parse span events to reconstruct the execution flow:
# user prompt -> LLM call -> tool selection -> tool execution -> response

print("=" * 60)
print("TRACE EXECUTION FLOW")
print("=" * 60)

def parse_span_flow(spans):
    """
    Parse spans to reconstruct the execution flow of an agent invocation.
    Returns a structured list of steps: prompt, llm_call, tool_selection, tool_execution, response.
    """
    flow_steps = []
    
    for span in spans:
        span_name = span.get('name', '')
        attrs = span.get('attributes', {})
        events = span.get('events', [])
        start_ns = span.get('startTimeUnixNano', 0)
        end_ns = span.get('endTimeUnixNano', 0)
        duration_ms = (end_ns - start_ns) / 1_000_000 if start_ns and end_ns else 0
        
        # Classify span type based on attributes and name
        gen_ai_system = attrs.get('gen_ai.system', '')
        gen_ai_operation = attrs.get('gen_ai.operation.name', '')
        
        step = {
            'span_name': span_name,
            'span_id': span.get('spanId', ''),
            'trace_id': span.get('traceId', ''),
            'duration_ms': round(duration_ms, 2),
            'events': []
        }
        
        # Parse events within the span
        for event in events:
            event_name = event.get('name', '')
            event_attrs = event.get('attributes', {})
            
            if 'gen_ai.user.message' in event_name:
                step['type'] = 'user_prompt'
                content = event_attrs.get('gen_ai.content', '')
                step['content'] = content[:200] if content else ''
            elif 'gen_ai.assistant.message' in event_name:
                step['type'] = 'llm_response'
                content = event_attrs.get('gen_ai.content', '')
                step['content'] = content[:200] if content else ''
            elif 'gen_ai.tool.message' in event_name:
                step['type'] = 'tool_result'
                content = event_attrs.get('gen_ai.content', '')
                step['content'] = content[:200] if content else ''
            
            step['events'].append({
                'name': event_name,
                'attributes': {k: str(v)[:100] for k, v in event_attrs.items()} if event_attrs else {}
            })
        
        # Classify by gen_ai attributes if not already classified by events
        if 'type' not in step:
            if gen_ai_operation == 'chat':
                step['type'] = 'llm_call'
            elif 'tool' in span_name.lower():
                step['type'] = 'tool_execution'
            else:
                step['type'] = 'agent_span'
        
        flow_steps.append(step)
    
    return flow_steps


if recent_traces:
    # Group spans by trace ID and show the flow for the most recent trace
    traces_by_id = {}
    for span in recent_traces:
        tid = span.get('traceId', 'unknown')
        if tid not in traces_by_id:
            traces_by_id[tid] = []
        traces_by_id[tid].append(span)
    
    # Pick the trace with the most spans (likely the most complete)
    most_complete_trace_id = max(traces_by_id, key=lambda k: len(traces_by_id[k]))
    trace_spans = traces_by_id[most_complete_trace_id]
    
    print(f"\nTrace ID: {most_complete_trace_id}")
    print(f"Total spans: {len(trace_spans)}")
    print(f"\nExecution Flow:")
    print("-" * 60)
    
    flow = parse_span_flow(trace_spans)
    
    # Display the flow with indentation showing the execution sequence
    step_icons = {
        'user_prompt': '[USER]     ',
        'llm_call': '[LLM]      ',
        'llm_response': '[LLM RESP] ',
        'tool_execution': '[TOOL EXEC]',
        'tool_result': '[TOOL RESP]',
        'agent_span': '[AGENT]    ',
    }
    
    for i, step in enumerate(flow, 1):
        step_type = step.get('type', 'unknown')
        icon = step_icons.get(step_type, '[???]      ')
        print(f"\n  {i}. {icon} {step['span_name']}")
        print(f"     Duration: {step['duration_ms']}ms")
        if step.get('content'):
            print(f"     Content: {step['content'][:120]}...")
        if step.get('events'):
            print(f"     Events: {len(step['events'])}")
            for evt in step['events'][:3]:
                print(f"       - {evt['name']}")
else:
    print("\nNo traces available. Run Step 4 to generate test data first.")

In [ ]:
# Extract tool calls and parameters from trace events
print("=" * 60)
print("TOOL CALLS AND PARAMETERS FROM TRACES")
print("=" * 60)

def extract_tool_calls(spans):
    """
    Extract tool call information from trace spans.
    
    Tool calls in Strands agents appear in multiple places:
    1. gen_ai.assistant / gen_ai.assistant.message events with tool_use content
       (in any scope, but especially opentelemetry.instrumentation.botocore.bedrock-runtime)
    2. gen_ai.tool / gen_ai.tool.message events with tool results
    3. body.output.messages in ADOT event records (strands.telemetry.tracer scope)
    """
    tool_calls = []
    
    for span in spans:
        events = span.get('events', [])
        attrs = span.get('attributes', {})
        span_name = span.get('name', '')
        scope_name = span.get('scope', {}).get('name', '')
        
        # ── Method 1: Parse events within spans ──
        for event in events:
            event_name = event.get('name', '')
            event_attrs = event.get('attributes', {})
            
            # Look for tool_use in assistant messages (both .message and plain)
            if 'gen_ai.assistant' in event_name:
                content = event_attrs.get('gen_ai.content', '')
                if isinstance(content, str) and content.strip():
                    try:
                        content_parsed = json.loads(content)
                        items = content_parsed if isinstance(content_parsed, list) else [content_parsed]
                        for item in items:
                            if isinstance(item, dict) and item.get('type') == 'tool_use':
                                tool_calls.append({
                                    'tool_name': item.get('name', 'unknown'),
                                    'parameters': item.get('input', {}),
                                    'span_name': span_name,
                                    'scope': scope_name,
                                    'trace_id': span.get('traceId', ''),
                                })
                    except (json.JSONDecodeError, TypeError):
                        pass
            
            # Look for tool results
            if 'gen_ai.tool' in event_name:
                tool_name = event_attrs.get('gen_ai.tool.name', '')
                tool_call_id = event_attrs.get('gen_ai.tool.call.id', '')
                content = event_attrs.get('gen_ai.content', '')
                
                if tool_name:
                    matched = False
                    for tc in tool_calls:
                        if tc['tool_name'] == tool_name and 'result' not in tc:
                            tc['result'] = content[:300] if content else ''
                            tc['tool_call_id'] = tool_call_id
                            matched = True
                            break
                    if not matched:
                        tool_calls.append({
                            'tool_name': tool_name,
                            'tool_call_id': tool_call_id,
                            'result': content[:300] if content else '',
                            'span_name': span_name,
                            'scope': scope_name,
                            'trace_id': span.get('traceId', ''),
                        })
        
        # ── Method 2: Parse body for tool calls (ADOT event format) ──
        body = span.get('body', {})
        if isinstance(body, dict):
            output = body.get('output', {})
            if isinstance(output, dict):
                messages = output.get('messages', [])
                if isinstance(messages, list):
                    for msg in messages:
                        content = msg.get('content', {})
                        if isinstance(content, dict) and content.get('type') == 'tool_use':
                            tool_calls.append({
                                'tool_name': content.get('name', 'unknown'),
                                'parameters': content.get('input', {}),
                                'span_name': span_name,
                                'scope': scope_name,
                                'trace_id': span.get('traceId', ''),
                                'source': 'body',
                            })
                        elif isinstance(content, list):
                            for item in content:
                                if isinstance(item, dict) and item.get('type') == 'tool_use':
                                    tool_calls.append({
                                        'tool_name': item.get('name', 'unknown'),
                                        'parameters': item.get('input', {}),
                                        'span_name': span_name,
                                        'scope': scope_name,
                                        'trace_id': span.get('traceId', ''),
                                        'source': 'body',
                                    })
    
    return tool_calls


if recent_traces:
    all_tool_calls = extract_tool_calls(recent_traces)
    
    if all_tool_calls:
        print(f"\nExtracted {len(all_tool_calls)} tool call(s):\n")
        for i, tc in enumerate(all_tool_calls, 1):
            print(f"  Tool Call {i}:")
            print(f"    Tool: {tc['tool_name']}")
            if tc.get('parameters'):
                params_str = json.dumps(tc['parameters'], default=str)
                print(f"    Parameters: {params_str[:200]}")
            if tc.get('result'):
                print(f"    Result: {tc['result'][:150]}...")
            print(f"    Scope: {tc.get('scope', 'N/A')}")
            print(f"    Trace: {tc['trace_id'][:30]}...")
            print()
    else:
        print("\nNo tool calls extracted from traces.")
        print("This can happen if:")
        print("  - The queried traces don't contain tool invocations")
        print("  - Tool call events are in a scope not yet queried")
        print("  - Events haven't propagated to CloudWatch yet")
        print()
        print("TIP: Run Step 4 to generate traces with tool calls,")
        print("then re-run this cell after waiting for log propagation.")
else:
    print("\nNo traces available. Run Step 8 first to fetch traces.")

## Step 7: CloudWatch Custom Dashboard

A **Single Pane of Glass** dashboard consolidates all agent observability and evaluation metrics into one view. This dashboard provides:

- **Operational metrics**: Invocation counts, latency, error rates
- **Evaluation scores**: Goal success, tool accuracy, RBAC compliance
- **Debugging aids**: Recent failures, tool usage breakdown, drift detection

The dashboard uses CloudWatch metrics from both the OTEL instrumentation (Module 3) and the evaluation pipeline (this module).

In [ ]:
# Build the CloudWatch Dashboard JSON body
# Operational metrics: AWS/Bedrock-AgentCore namespace (auto-published by AgentCore Runtime)
# Evaluation metrics: Logs Insights queries against eval results log group (no CW Metrics auto-published)

DASHBOARD_NAME = "EcommerceWorkshop-ProductCatalogAgent"

# Correct namespace for AgentCore Runtime operational metrics
AGENT_NAMESPACE = "AWS/Bedrock-AgentCore"

# Evaluation results are in CloudWatch Logs, not Metrics
EVAL_LOG_GROUP = (
    f"/aws/bedrock-agentcore/evaluations/results/{ONLINE_EVAL_CONFIG.get('config_id', 'unknown')}"
    if ONLINE_EVAL_CONFIG else None
)
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT" if AGENT_ID else None

# Agent Runtime dimension format: "agent_name::endpoint_name"
AGENT_DIMENSION_NAME = f"{AGENT_NAME}::DEFAULT" if AGENT_NAME else "unknown"

widgets = []
y_pos = 0

# ── Row 1: Agent Invocations + Latency (from AWS/Bedrock-AgentCore metrics) ──
widgets.append({
    "type": "metric",
    "x": 0, "y": y_pos, "width": 12, "height": 6,
    "properties": {
        "title": "Agent Invocations Over Time",
        "view": "timeSeries",
        "stacked": False,
        "metrics": [
            [AGENT_NAMESPACE, "Invocations", "Name", AGENT_DIMENSION_NAME,
             "Operation", "InvokeAgentRuntime", {"stat": "Sum"}]
        ],
        "region": REGION, "period": 300,
        "yAxis": {"left": {"min": 0, "label": "Count"}},
    },
})
widgets.append({
    "type": "metric",
    "x": 12, "y": y_pos, "width": 12, "height": 6,
    "properties": {
        "title": "Agent Latency (ms)",
        "view": "timeSeries",
        "stacked": False,
        "metrics": [
            [AGENT_NAMESPACE, "Latency", "Name", AGENT_DIMENSION_NAME,
             "Operation", "InvokeAgentRuntime", {"stat": "Average"}],
            [AGENT_NAMESPACE, "Latency", "Name", AGENT_DIMENSION_NAME,
             "Operation", "InvokeAgentRuntime", {"stat": "p99", "label": "p99 Latency"}],
        ],
        "region": REGION, "period": 300,
        "yAxis": {"left": {"min": 0, "label": "Milliseconds"}},
    },
})
y_pos += 6

# ── Row 2: Evaluation Scores from Logs Insights ──
if EVAL_LOG_GROUP:
    widgets.append({
        "type": "log",
        "x": 0, "y": y_pos, "width": 12, "height": 6,
        "properties": {
            "title": "Evaluation Scores Over Time",
            "query": (
                f"SOURCE '{EVAL_LOG_GROUP}'\n"
                "| fields @timestamp, evaluatorName, value, label\n"
                "| filter ispresent(evaluatorName)\n"
                "| sort @timestamp desc\n"
                "| limit 50"
            ),
            "region": REGION, "view": "table",
        },
    })
    widgets.append({
        "type": "log",
        "x": 12, "y": y_pos, "width": 12, "height": 6,
        "properties": {
            "title": "Average Score by Evaluator",
            "query": (
                f"SOURCE '{EVAL_LOG_GROUP}'\n"
                "| filter ispresent(evaluatorName)\n"
                "| stats avg(value) as avg_score, count(*) as eval_count by evaluatorName\n"
                "| sort avg_score asc"
            ),
            "region": REGION, "view": "table",
        },
    })
    y_pos += 6

# ── Row 3: Low Scores + Score Distribution ──
if EVAL_LOG_GROUP:
    widgets.append({
        "type": "log",
        "x": 0, "y": y_pos, "width": 12, "height": 6,
        "properties": {
            "title": "Low Evaluation Scores (< 0.5)",
            "query": (
                f"SOURCE '{EVAL_LOG_GROUP}'\n"
                "| fields @timestamp, evaluatorName, value, label, explanation\n"
                "| filter value < 0.5\n"
                "| sort @timestamp desc\n"
                "| limit 20"
            ),
            "region": REGION, "view": "table",
        },
    })
    widgets.append({
        "type": "log",
        "x": 12, "y": y_pos, "width": 12, "height": 6,
        "properties": {
            "title": "Score Distribution by Label",
            "query": (
                f"SOURCE '{EVAL_LOG_GROUP}'\n"
                "| filter ispresent(label)\n"
                "| stats count(*) as count by label\n"
                "| sort count desc"
            ),
            "region": REGION, "view": "bar",
        },
    })
    y_pos += 6

# ── Row 4: Error Rate + Duration (from AWS/Bedrock-AgentCore metrics) ──
widgets.append({
    "type": "metric",
    "x": 0, "y": y_pos, "width": 12, "height": 6,
    "properties": {
        "title": "Error Rate",
        "view": "timeSeries",
        "stacked": False,
        "metrics": [
            [AGENT_NAMESPACE, "Errors", "Name", AGENT_DIMENSION_NAME,
             "Operation", "InvokeAgentRuntime", {"stat": "Sum", "label": "Errors"}],
            [AGENT_NAMESPACE, "Invocations", "Name", AGENT_DIMENSION_NAME,
             "Operation", "InvokeAgentRuntime", {"stat": "Sum", "label": "Total Invocations"}],
        ],
        "region": REGION, "period": 300,
        "yAxis": {"left": {"min": 0, "label": "Count"}},
    },
})
widgets.append({
    "type": "metric",
    "x": 12, "y": y_pos, "width": 12, "height": 6,
    "properties": {
        "title": "Agent Duration Distribution",
        "view": "timeSeries",
        "stacked": False,
        "metrics": [
            [AGENT_NAMESPACE, "Duration", "Name", AGENT_DIMENSION_NAME,
             "Operation", "InvokeAgentRuntime", {"stat": "Average", "label": "Avg Duration"}],
            [AGENT_NAMESPACE, "Duration", "Name", AGENT_DIMENSION_NAME,
             "Operation", "InvokeAgentRuntime", {"stat": "p99", "label": "p99 Duration"}],
        ],
        "region": REGION, "period": 300,
        "yAxis": {"left": {"min": 0, "label": "Milliseconds"}},
    },
})
y_pos += 6

# ── Row 5: Tool calls from traces (Logs Insights) ──
if RUNTIME_LOG_GROUP:
    widgets.append({
        "type": "log",
        "x": 0, "y": y_pos, "width": 24, "height": 6,
        "properties": {
            "title": "Recent Agent Traces (Tool Calls)",
            "query": (
                f"SOURCE '{RUNTIME_LOG_GROUP}'\n"
                "| fields @timestamp, name, attributes.gen_ai.operation.name as operation\n"
                "| filter ispresent(name)\n"
                "| sort @timestamp desc\n"
                "| limit 30"
            ),
            "region": REGION, "view": "table",
        },
    })
    y_pos += 6

dashboard_body = {"widgets": widgets}
dashboard_json = json.dumps(dashboard_body)

print("Dashboard Configuration Built")
print(f"  Dashboard name: {DASHBOARD_NAME}")
print(f"  Total widgets: {len(widgets)}")
print(f"  Dashboard JSON size: {len(dashboard_json)} bytes")
print(f"\nWidget Layout:")
print(f"  Row 1: Agent Invocations (metric) | Agent Latency (metric)")
print(f"  Row 2: Evaluation Scores (log query) | Average by Evaluator (log query)")
print(f"  Row 3: Low Scores (log query) | Score Distribution (log query)")
print(f"  Row 4: Error Rate (metric) | Duration Distribution (metric)")
print(f"  Row 5: Recent Traces (log query)")
print(f"\nMetric namespace: {AGENT_NAMESPACE}")
print(f"Eval log group: {EVAL_LOG_GROUP}")


In [ ]:
# Deploy the dashboard to CloudWatch
cloudwatch = boto3.client('cloudwatch', region_name=REGION)

print("=" * 60)
print("DEPLOYING CLOUDWATCH DASHBOARD")
print("=" * 60)

try:
    response = cloudwatch.put_dashboard(
        DashboardName=DASHBOARD_NAME,
        DashboardBody=dashboard_json
    )
    
    # Check for validation messages
    messages = response.get('DashboardValidationMessages', [])
    
    if not messages:
        print(f"\nDashboard deployed successfully: {DASHBOARD_NAME}")
    else:
        print(f"\nDashboard deployed with validation messages:")
        for msg in messages:
            print(f"  - {msg.get('Message', 'Unknown message')}")
            print(f"    Data Path: {msg.get('DataPath', 'N/A')}")

except Exception as e:
    print(f"\nError deploying dashboard: {e}")
    print("You may need cloudwatch:PutDashboard permissions.")

In [ ]:
# Output the dashboard URL for quick access
dashboard_url = (
    f"https://{REGION}.console.aws.amazon.com/cloudwatch/home"
    f"?region={REGION}#dashboards/dashboard/{DASHBOARD_NAME}"
)

print("=" * 60)
print("DASHBOARD URL")
print("=" * 60)
print(f"\nOpen your dashboard in the AWS Console:")
print(f"\n  {dashboard_url}")
print(f"\nDashboard: {DASHBOARD_NAME}")
print(f"Region: {REGION}")
print(f"\nNote: Metrics will populate as agent invocations and evaluations generate data.")
print("It may take a few minutes for initial data points to appear.")

#### When to Use Online vs. Batch Evaluation

| Approach | Use Case | Cost | Latency | Coverage |
|----------|----------|------|---------|----------|
| **Online** (Module 04) | Continuous quality monitoring | Per-invocation LLM cost | Real-time | Sampled (10–100% of traffic) |
| **Batch** (Module 05) | Deep analysis, drift detection | Batch LLM cost | Hours | Full coverage of selected window |

**Best practice:** Use online evaluation as your early warning system — it samples production traffic and alerts on quality drops. Use batch evaluation for root-cause analysis — when online eval flags a problem, run batch eval on the full trace set to understand scope and severity.

Together, they form the **continuous monitoring loop:**
1. Online eval detects drift → 2. Batch eval quantifies impact → 3. Failed traces become new test cases → 4. Module 02 baseline is enriched → 5. Agent is improved → back to 1.

## Step 8: Summary

In [ ]:
print("\n" + "=" * 70)
print("MODULE 4 COMPLETE: AGENTCORE EVALUATIONS")
print("=" * 70)

print("\n1. SETUP")
print(f"   Evaluation IAM role: {EVALUATION_ROLE_NAME}")
print(f"   Product Catalog Agent: {AGENT_NAME if AGENT_INFO else 'NOT FOUND'}")

print("\n2. ONLINE EVALUATION")
if ONLINE_EVAL_CONFIG:
    print(f"   Config: {ONLINE_EVAL_CONFIG.get('config_name')}")
    print(f"   Config ID: {ONLINE_EVAL_CONFIG.get('config_id')}")
    print(f"   Status: {ONLINE_EVAL_CONFIG.get('status')}")
    print(f"   Evaluators: Helpfulness, GoalSuccessRate, ToolSelectionAccuracy, Coherence")
else:
    print("   Not configured")

print("\n3. TEST INVOCATIONS")
successful_invocations = sum(1 for r in INVOCATION_RESULTS if r.get('success'))
print(f"   Invocations: {successful_invocations}/{len(INVOCATION_RESULTS)} successful")
print(f"   Sessions collected: {len(SESSION_IDS)}")

print("\n4. TRACE EXPLORATION")
if recent_traces:
    print(f"   GenAI trace spans collected: {len(recent_traces)}")
    print(f"   Unique traces: {len(set(t.get('traceId', '') for t in recent_traces))}")
else:
    print("   No traces explored (run Step 6)")

print("\n5. CLOUDWATCH DASHBOARD")
print(f"   Dashboard: {DASHBOARD_NAME}")
dashboard_url = (
    f"https://{REGION}.console.aws.amazon.com/cloudwatch/home"
    f"?region={REGION}#dashboards/dashboard/{DASHBOARD_NAME}"
)
print(f"   URL: {dashboard_url}")

print("\n" + "=" * 70)
print("KEY TAKEAWAYS")
print("=" * 70)
print("""
1. Online Evaluation:
   - Continuously monitors agent quality with live traffic
   - Results automatically stored in CloudWatch
   - Scores published as CloudWatch metrics for alarming

2. Trace Exploration:
   - OTEL traces capture the full execution flow
   - Parse span events to see: prompt -> LLM call -> tool -> response
   - Extract tool calls and parameters for debugging

3. Built-in Evaluators:
   - Session-level: GoalSuccessRate
   - Trace-level: Helpfulness, Coherence, Correctness, etc.
   - Tool-level: ToolSelectionAccuracy, ToolParameterAccuracy

4. CloudWatch Dashboard:
   - Single pane of glass for operations + evaluations
   - Drift detection with baseline thresholds
   - RBAC compliance monitoring and role-based invocation tracking

5. Integration with Batch Evaluation (Module 05):
   - Module 05 extracts OTEL events directly from CloudWatch logs
   - Runs offline evaluators for drift detection and feedback loops
   - On-demand evaluation is not used due to scope mismatch with runtime OTEL instrumentation
""")
print("=" * 70)

In [ ]:
# Store variables for potential use in other modules
%store ONLINE_EVAL_CONFIG
%store EVALUATION_ROLE_ARN
%store SESSION_IDS
%store ON_DEMAND_RESULTS
%store DASHBOARD_NAME

print("Variables stored for use in subsequent modules")

---

## Cleanup (Optional)

Run the cell below to delete evaluation resources. **Only run this when you're done with the workshop.**

In [ ]:
# UNCOMMENT AND RUN TO CLEAN UP EVALUATION RESOURCES

# def cleanup_evaluation_resources(delete_config=False, delete_role=False):
#     """Clean up evaluation resources."""
#     print("\nCLEANUP - This will delete resources!")
#
#     if delete_config and ONLINE_EVAL_CONFIG:
#         config_id = ONLINE_EVAL_CONFIG.get('config_id')
#         print(f"\nDeleting online evaluation config: {config_id}")
#         try:
#             agentcore_control_client.delete_online_evaluation_config(
#                 onlineEvaluationConfigId=config_id
#             )
#             print("Config deleted")
#         except Exception as e:
#             print(f"Error: {e}")
#
#     if delete_role:
#         print(f"\nDeleting IAM role: {EVALUATION_ROLE_NAME}")
#         try:
#             policies = iam_client.list_role_policies(RoleName=EVALUATION_ROLE_NAME)
#             for policy_name in policies.get('PolicyNames', []):
#                 iam_client.delete_role_policy(
#                     RoleName=EVALUATION_ROLE_NAME,
#                     PolicyName=policy_name
#                 )
#             iam_client.delete_role(RoleName=EVALUATION_ROLE_NAME)
#             print("Role deleted")
#         except Exception as e:
#             print(f"Error: {e}")
#
# cleanup_evaluation_resources(delete_config=True, delete_role=True)

---

## Module 4 Summary

You set up **AgentCore Evaluations** to automatically assess the Product Catalog Agent's performance, explored OTEL traces, and built a custom CloudWatch dashboard.

### What Was Created

| Component | Description |
|-----------|-------------|
| **Evaluation IAM Role** | Permissions for Bedrock model access and CloudWatch read/write |
| **Online Evaluation Config** | Continuous monitoring with Helpfulness, GoalSuccessRate, ToolSelectionAccuracy, Coherence |
| **On-Demand Evaluations** | Targeted assessment of specific sessions and traces |
| **CloudWatch Dashboard** | Single pane of glass with operations, evaluations, and drift detection |

### Evaluation Types

| Type | How It Works | When to Use |
|------|-------------|-------------|
| **Online** | Samples live traffic, runs LLM-as-Judge continuously | Production quality monitoring |
| **On-Demand** | Downloads spans, calls Evaluate API programmatically | Issue investigation, testing evaluators |

### Trace Exploration

| Component | What It Shows |
|-----------|---------------|
| **Span Flow** | user prompt -> LLM call -> tool selection -> tool execution -> response |
| **Tool Extraction** | Tool names, parameters, and results from GenAI events |
| **Trace Grouping** | Spans grouped by trace ID for full invocation view |

### CloudWatch Dashboard Widgets

| Row | Left Widget | Right Widget |
|-----|-------------|--------------|
| 1 | Agent Invocations Over Time | Average Latency |
| 2 | Goal Success Rate | Tool Selection Accuracy |
| 3 | RBAC Compliance Score | Evaluation Score Distribution |
| 4 | Recent Evaluation Failures | Tool Usage Breakdown |
| 5 | Error Rate | Batch Eval Drift Detection |
| 6 | Invocations by Role | |

### Integration with Module 3 Observability

| Module 3 Component | Module 4 Usage |
|-------------------|----------------|
| OTEL traces (X-Ray) | Data source for online evaluation |
| GenAI events (vendedlogs) | Span data for on-demand evaluation |
| OTEL service name | Filtering traces to this specific agent |
| CloudWatch Log Delivery | Routes telemetry data to evaluation pipeline |

### Workshop Complete!

You have now completed the full agent development lifecycle:
1. **Module 01**: Built a local prototype with golden dataset
2. **Module 02**: Established offline evaluation baseline
3. **Module 03**: Deployed to production with RBAC and observability
4. **Module 04**: Set up online and on-demand evaluation, trace exploration, and a custom CloudWatch dashboard for continuous quality monitoring